In [52]:
# Import Libraries:
import warnings
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from marvin import config
from marvin.tools.query import Query
from marvin.tools.cube import Cube
from marvin.tools.maps import Maps

# Ignore warnings, and configure Marvin:
warnings.simplefilter("ignore")
config.setRelease("DR17")
config.mode = "remote"

# Import and extract data from OTTER table:
otter_cands_data_raw = pd.read_table("OtterCands.csv", delimiter=",")
otter_cands_data_name = otter_cands_data_raw["Name"]
otter_cands_data_z = otter_cands_data_raw["redshift"]
otter_cands_data_conf = otter_cands_data_raw["confidence"]
otter_cands_data_ra = otter_cands_data_raw["ra"]
otter_cands_data_dec = otter_cands_data_raw["dec"]
otter_cands_data_has_host_data = otter_cands_data_raw["Host info in OTTER?"]

otter_cands_array_name = otter_cands_data_name.to_numpy()
otter_cands_array_z = otter_cands_data_z.to_numpy()
otter_cands_array_conf = otter_cands_data_conf.to_numpy()
otter_cands_array_ra = otter_cands_data_ra.to_numpy()
otter_cands_array_dec = otter_cands_data_dec.to_numpy()
otter_cands_array_has_host_data = otter_cands_data_has_host_data.to_numpy()

# Display OTTER Candidates:
print("OTTER Candidates:")
display(otter_cands_data_raw)

# Create Search Query for Marvin:
def marvin_search(ra_deg, dec_deg, z, search_radius_arcsec=0, search_depth_z=0):
    """
    Description:
        Creates a search query for marvin to take in and use for MaNGA. This is for finding an object spatially only.

    Arguments:
        ra_deg -> The right ascension coordinate of the object in degrees (Integer and Float).
        dec_deg -> The declination coordinate of the object in degrees (Integer and Float).
        z -> The redshift of the object (Integer and Float).
        search_radius_arcsec -> The search radius around the object's coordinates in arcseconds (Integer of Float).
        search_depth_z -> The search depth around the object's coordinates in arcseconds (Integer of Float).

    Returns:
        A query statement that can be used for the query of marvin, this is a string.
    """

    search_radius_deg = (search_radius_arcsec * u.arcsec).to(u.deg).value #Converts arcsecs to degrees.
    search_req = (f"radial({ra_deg}, {dec_deg}, {search_radius_deg}) and "
                  f"nsa.z < {z + search_depth_z} and nsa.z > {z - search_depth_z}") #Creates search requirement.
    return search_req

# Controls for search:
search_radius =  5 #Arcsecs
redshift_depth = 0.5 #z

# Loop through each TDE candidate provided from OTTER:
results_all = []

for z, ra, dec, name in zip(otter_cands_array_z, otter_cands_array_ra, otter_cands_array_dec, otter_cands_array_name):
    return_vals = ["nsa.z", "nsa.sersic_mass", "nsa.sersic_logmass", "nsa.ra", "nsa.dec", "cube.ra", "cube.dec"]
    q = Query(search_filter=marvin_search(ra, dec, z, search_radius, redshift_depth), return_params=return_vals)
    r = q.run()

    if (len(r.results) > 0):
        results_input_df = r.toDF()
        results_input_df["Candidate Name"] = name
        results_all.append(results_input_df)

# Add in the extra columns for the overall results table:
if (len(results_all) > 0):
    results_final_df = pd.concat(results_all, ignore_index=True)
else:
    results_final_df = pd.DataFrame()

# Display results:
print("Results (From MaNGA):")
display(results_final_df)

print("Total Galaxies Returned (from MaNGA):",  len(set(results_final_df["Candidate Name"])))
print(f"Search Radius: {search_radius} arcsec(s)")
print(f"Redshift Search Depth: {redshift_depth} z")

OTTER Candidates:


,Name,redshift,confidence,ra,dec,Host info in OTTER?,Bibcode
0,NGC 5905,0.011240,3.1,228.847167,55.517108,True,"['1996A&A...309L..35B', '2017ApJ...838..149A',..."
1,ARP 299-B AT1,0.010411,3.3,172.129115,58.561329,False,['2018Sci...361..482M']
2,XMMSL1 J0740-85,0.017300,3.3,115.034125,-85.658681,True,"['2017A&A...598A..29S', '2017ApJ...835...64G',..."
3,XMMSL2 J144605.0+685735,0.029000,3.3,221.520833,68.958611,True,"['2019A&A...630A..98S', '2023PASP..135c4101G',..."
4,2016fnl,0.016300,3.3,7.487667,32.893750,True,"['2016ATel.9433....1G', '2016TNSCR.615....1Y',..."
5,2022gri,0.028000,3.3,109.586643,33.994853,False,"['2022TNSAN..99....1Y', '2022TNSCR1156....1Y',..."
6,2022dsb,0.023000,3.3,235.590633,-22.670580,False,"['2022TNSCR.584....1F', '2024ApJ...974..241A',..."
7,2019azh,0.022000,3.3,123.320627,22.648301,True,"['2019TNSCR.287....1B', '2019TNSTR...2....1S',..."
8,2020wey,0.027410,3.3,136.357958,61.802538,False,"['2020TNSCR3228....1A', '2023ApJ...942....9H',..."
9,AT2019ahk,0.026200,3.3,105.047461,-66.040214,False,"['2019ApJ...883..111H', '2024ApJ...974..241A',..."


Results (From MaNGA):


,mangaid,plateifu,z,sersic_mass,sersic_logmass,nsa_ra,nsa_dec,cube_ra,cube_dec,Candidate Name
0,1-298319,9495-3702,0.022224,4.460553e+09,9.649389,123.320626,22.648319,123.320626,22.648319,2019azh


Total Galaxies Returned (from MaNGA): 1
Search Radius: 5 arcsec(s)
Redshift Search Depth: 0.5 z
